# MDP YOLOv5 Training Notebook
This notebook trains a YOLOv5 model for MDP symbols. It references the architecture YAMLs in `C:\github\MDP-Algorithm\models` so your Algorithm server and training stay consistent.

In [ ]:
# Environment check and paths
from pathlib import Path
import sys, os
MDP_ALGO_ROOT = Path(r'c:\github\MDP-Algorithm')
assert MDP_ALGO_ROOT.exists(), f'MDP-Algorithm repo not found at {MDP_ALGO_ROOT}'
print('Python', sys.version)
try:
    import torch
    print('Torch', torch.__version__, 'CUDA available:', torch.cuda.is_available())
except Exception as e:
    print('Torch not installed yet:', e)

# Project paths
PROJ_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name.lower()=='notebooks' else Path.cwd()
DATA_DIR = PROJ_ROOT / 'data'
RUNS_DIR = PROJ_ROOT / 'runs'
print('Project root:', PROJ_ROOT)
print('MDP_ALGO_ROOT:', MDP_ALGO_ROOT)

# Add Algorithm repo to sys.path to access consts and name mappings
if str(MDP_ALGO_ROOT) not in sys.path:
    sys.path.insert(0, str(MDP_ALGO_ROOT))
from consts import NAME_TO_ID
# Build class names from consts, excluding 'NA'
names = [k for k,_ in sorted(((k,v) for k,v in NAME_TO_ID.items() if k!='NA'), key=lambda kv: kv[1])]
len(names), names[:10]

## (Optional) Install dependencies into this venv
Run this once per environment. Requires Git for YOLOv5 install.

In [ ]:
# If running in this kernel, you can install here.
# Uncomment to install. Otherwise, install via `pip install -r requirements.txt` in PowerShell.
# %pip install -r ../requirements.txt
# import importlib; importlib.reload(site)

## Create/Update dataset YAML
Set `train` and `val` to your image folder roots. Labels must be in YOLO format.

In [ ]:
import yaml
dataset_yaml = DATA_DIR / 'mdp.yaml'
mdp = {
    'path': str(DATA_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(names),
    'names': names,
}
dataset_yaml.parent.mkdir(parents=True, exist_ok=True)
with open(dataset_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(mdp, f, sort_keys=False, allow_unicode=True)
print('Wrote', dataset_yaml)
print(dataset_yaml.read_text())

## Choose model architecture from MDP-Algorithm/models
`yolov5n.yaml` is the lightest (fastest). You can swap to `yolov5s.yaml` etc.

In [ ]:
CFG = (MDP_ALGO_ROOT / 'models' / 'yolov5n.yaml').resolve()
assert CFG.exists(), CFG
CFG

## Train
This calls YOLOv5's training entrypoint. Adjust `epochs`, `batch`, `imgsz` as needed.

In [ ]:
# If `from yolov5 import train` fails, ensure requirements are installed in this kernel.
from yolov5 import train
run_name = 'mdp_yolov5n'
train.run(
    data=str(dataset_yaml),
    cfg=str(CFG),
    weights='',  # start from scratch; provide a .pt path to fine-tune
    epochs=50,
    batch=16,
    imgsz=640,
    project=str(RUNS_DIR / 'train'),
    name=run_name,
    exist_ok=True,
)

## Locate best weights

In [ ]:
from pathlib import Path
wdir = RUNS_DIR / 'train' / 'mdp_yolov5n' / 'weights'
best = (wdir / 'best.pt') if (wdir / 'best.pt').exists() else None
best

## (Optional) Copy to Algorithm repo and quick sanity-check inference

In [ ]:
import shutil, torch
if best and best.exists():
    target = MDP_ALGO_ROOT / 'MDP_custom.pt'
    shutil.copy(best, target)
    print('Copied to', target)
    # Load via hubconf in MDP-Algorithm for consistency with server inference
    model = torch.hub.load(str(MDP_ALGO_ROOT), 'custom', path=str(target), source='local')
    print('Model ready for inference via hubconf')
else:
    print('No best.pt found yet')
